In [1]:
# ============================================================
# CARDIOVISION AI
# CCA 3D CCTA CORONARY ARTERY SEGMENTATION v2
#
# FIXED RESEARCH PIPELINE
#
# Major fixes versus v1
# 1. Patient/case-level split
# 2. Training uses foreground + hard-negative + background patches
# 3. Each case is loaded once per training epoch
# 4. Validation uses FULL-VOLUME sliding-window inference
# 5. Test uses the identical full-volume inference protocol
# 6. Validation threshold is optimized, never optimized on test
# 7. Tversky + Dice + BCE loss for severe class imbalance
# 8. Physical-space resampling to common spacing
# 9. HD95 is reported in millimetres
# 10. Non-inplace activations so Grad-CAM works
# 11. Grad-CAM is spatially aligned back into full-volume space
# 12. Best checkpoint stores the selected validation threshold
# 13. Per-case and aggregate metrics are saved
# 14. No full-volume dataset caching
# 15. No DataParallel with batch size 1
#
# Dataset:
#   MedHK23/CCA
#
# Recommended Kaggle accelerator:
#   2x Tesla T4
#
# GPU policy:
#   GPU 0 is used for the main experiment.
#   GPU 1 can be used for a separate experiment.
#
# ============================================================

import os
import gc
import sys
import math
import time
import random
import warnings
import subprocess
from pathlib import Path

warnings.filterwarnings("ignore")

# CUDA allocator must be configured before CUDA initialization.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42


def pip_install(package):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", package
    ])


# ============================================================
# IMPORTS
# ============================================================

try:
    import nibabel as nib
    from nibabel.processing import resample_from_to
except Exception:
    pip_install("nibabel")
    import nibabel as nib
    from nibabel.processing import resample_from_to

try:
    import scipy
    from scipy.ndimage import (
        binary_erosion,
        distance_transform_edt,
    )
except Exception:
    pip_install("scipy")
    from scipy.ndimage import (
        binary_erosion,
        distance_transform_edt,
    )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True


# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("=" * 80)
print("CARDIOVISION AI | CCA 3D CCTA SEGMENTATION v2")
print("=" * 80)

print("\nDEVICE")
print("-" * 80)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Selected device:", DEVICE)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Restart Kaggle with GPU enabled."
    )

print("GPU count:", torch.cuda.device_count())

for gpu_id in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(gpu_id)
    print(
        f"GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)} | "
        f"{props.total_memory / 1024**3:.2f} GB"
    )


# ============================================================
# CONFIGURATION
# ============================================================

DATASET_NAME = "MedHK23/CCA"

OUTPUT_ROOT = Path(
    "/kaggle/working/cardioVision_CCA_v2"
)

CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
RESULT_DIR = OUTPUT_ROOT / "results"
XAI_DIR = OUTPUT_ROOT / "xai"

for directory in [
    OUTPUT_ROOT,
    CHECKPOINT_DIR,
    RESULT_DIR,
    XAI_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# DATA / SPLIT
# ============================================================

TEST_FRACTION = 0.15
VAL_FRACTION = 0.15

# Fixed seed split for reproducibility.
# With 20 cases this gives 14/3/3.
TRAIN_PATCHES_PER_CASE = 16

# Number of patch batches generated from each case.
# Patches are generated in RAM only for the current case.
PATCH_BATCH_SIZE = 2


# ============================================================
# PREPROCESSING
# ============================================================

# Common physical spacing in mm.
# A moderate spacing avoids excessive memory growth.
TARGET_SPACING = (
    1.0,
    1.0,
    1.0
)

# Robust CCTA intensity window.
HU_MIN = -1000.0
HU_MAX = 1000.0


# ============================================================
# MODEL / TRAINING
# ============================================================

PATCH_SIZE = (
    96,
    96,
    96
)

BASE_CHANNELS = 16

EPOCHS = 15

LEARNING_RATE = 2e-4

WEIGHT_DECAY = 1e-5

PATIENCE = 4

GRAD_CLIP = 1.0

AMP_ENABLED = True


# ============================================================
# PATCH SAMPLING
# ============================================================

FOREGROUND_PATCH_FRACTION = 0.45

HARD_NEGATIVE_PATCH_FRACTION = 0.30

BACKGROUND_PATCH_FRACTION = 0.25

MIN_FOREGROUND_RATIO = 0.0005

HARD_NEGATIVE_MIN_DISTANCE = 3

HARD_NEGATIVE_MAX_DISTANCE = 24


# ============================================================
# INFERENCE
# ============================================================

INFERENCE_PATCH = PATCH_SIZE

INFERENCE_OVERLAP = 0.50

INFERENCE_BATCH_SIZE = 2

THRESHOLD_GRID = np.arange(
    0.10,
    0.91,
    0.05
)

DEFAULT_THRESHOLD = 0.50


# ============================================================
# FILES
# ============================================================

INDEX_CSV = OUTPUT_ROOT / "cardiovision_cca_index.csv"
SPLIT_CSV = OUTPUT_ROOT / "cardiovision_cca_split.csv"
HISTORY_CSV = RESULT_DIR / "training_history.csv"
VAL_METRICS_CSV = RESULT_DIR / "validation_metrics.csv"
TEST_METRICS_CSV = RESULT_DIR / "test_metrics.csv"
THRESHOLD_CSV = RESULT_DIR / "threshold_search.csv"

BEST_CHECKPOINT = (
    CHECKPOINT_DIR /
    "best_3d_unet_cca_v2.pth"
)

LATEST_CHECKPOINT = (
    CHECKPOINT_DIR /
    "latest_3d_unet_cca_v2.pth"
)


# ============================================================
# MEMORY
# ============================================================

def clear_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def gpu_memory():
    if not torch.cuda.is_available():
        return

    allocated = (
        torch.cuda.memory_allocated(0)
        / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved(0)
        / 1024**3
    )

    print(
        f"GPU memory | allocated={allocated:.3f} GB | "
        f"reserved={reserved:.3f} GB"
    )


# ============================================================
# GPU COMPUTE TEST
# ============================================================

print("\n" + "=" * 80)
print("GPU COMPUTE TEST")
print("=" * 80)

test_tensor = torch.randn(
    1,
    1,
    32,
    32,
    32,
    device=DEVICE
)

test_conv = nn.Conv3d(
    1,
    4,
    kernel_size=3,
    padding=1
).to(DEVICE)

with torch.no_grad():
    test_output = test_conv(test_tensor)

torch.cuda.synchronize()

assert test_tensor.device.type == "cuda"
assert next(test_conv.parameters()).device.type == "cuda"

print("Tensor device:", test_tensor.device)
print("Model device:", next(test_conv.parameters()).device)
print("Output:", tuple(test_output.shape))

del test_tensor
del test_conv
del test_output

clear_memory()

print("GPU computation verified.")


# ============================================================
# DATASET DOWNLOAD
# ============================================================

print("\n" + "=" * 80)
print("CCA DATASET")
print("=" * 80)

try:
    from huggingface_hub import snapshot_download
except Exception:
    pip_install("huggingface_hub")
    from huggingface_hub import snapshot_download

CACHE_DIR = Path(
    snapshot_download(
        repo_id=DATASET_NAME,
        repo_type="dataset"
    )
)

print("Repository:", CACHE_DIR)


# ============================================================
# NIFTI DISCOVERY
# ============================================================

print("\n" + "=" * 80)
print("DISCOVERING NIFTI FILES")
print("=" * 80)

image_files = sorted(
    CACHE_DIR.glob("train/images/*.nii.gz"),
    key=lambda x: int(x.name.split(".")[0])
)

label_files = sorted(
    CACHE_DIR.glob("train/labels/*.nii.gz"),
    key=lambda x: int(x.name.split(".")[0])
)

print("Images:", len(image_files))
print("Labels:", len(label_files))

if len(image_files) != len(label_files):
    raise RuntimeError("Image/label count mismatch.")

image_map = {
    int(p.name.split(".")[0]): p
    for p in image_files
}

label_map = {
    int(p.name.split(".")[0]): p
    for p in label_files
}

case_ids = sorted(
    set(image_map) & set(label_map)
)

print("Matched cases:", len(case_ids))

if len(case_ids) < 10:
    raise RuntimeError(
        "Too few matched CCA cases."
    )


# ============================================================
# DATASET VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("VALIDATING DATASET")
print("=" * 80)

records = []

for index, case_id in enumerate(
    case_ids,
    start=1
):

    image_path = image_map[case_id]
    label_path = label_map[case_id]

    print(
        f"Validating case {case_id} "
        f"({index}/{len(case_ids)})"
    )

    image_nii = nib.load(str(image_path))
    label_nii = nib.load(str(label_path))

    image_shape = tuple(image_nii.shape)
    label_shape = tuple(label_nii.shape)

    if image_shape != label_shape:
        raise RuntimeError(
            f"Shape mismatch case {case_id}: "
            f"{image_shape} vs {label_shape}"
        )

    image_spacing = tuple(
        np.asarray(
            image_nii.header.get_zooms()
        )[:3]
    )

    label_spacing = tuple(
        np.asarray(
            label_nii.header.get_zooms()
        )[:3]
    )

    if not np.allclose(
        image_spacing,
        label_spacing,
        atol=1e-3
    ):
        raise RuntimeError(
            f"Image/label spacing mismatch "
            f"case {case_id}"
        )

    label = np.asarray(
        label_nii.dataobj,
        dtype=np.uint8
    )

    unique_values = np.unique(label)

    if not np.all(
        np.isin(unique_values, [0, 1])
    ):
        raise RuntimeError(
            f"Invalid label values case {case_id}: "
            f"{unique_values}"
        )

    foreground_voxels = int(
        np.count_nonzero(label)
    )

    total_voxels = int(label.size)

    foreground_ratio = (
        foreground_voxels /
        max(total_voxels, 1)
    )

    records.append({
        "case_id": case_id,
        "image": str(image_path),
        "label": str(label_path),
        "shape": str(image_shape),
        "spacing": str(image_spacing),
        "foreground_voxels": foreground_voxels,
        "foreground_ratio": foreground_ratio,
    })

    del image_nii
    del label_nii
    del label

    clear_memory()


index_df = pd.DataFrame(records)

index_df.to_csv(
    INDEX_CSV,
    index=False
)

print("\nValidation complete.")
print("Index:", INDEX_CSV)

print(
    "Foreground ratio:",
    f"{index_df.foreground_ratio.min():.6f}",
    "to",
    f"{index_df.foreground_ratio.max():.6f}"
)


# ============================================================
# CASE-LEVEL SPLIT
# ============================================================

print("\n" + "=" * 80)
print("CASE-LEVEL SPLIT")
print("=" * 80)

rng = np.random.default_rng(SEED)

shuffled = np.array(
    sorted(case_ids)
)

rng.shuffle(shuffled)

n = len(shuffled)

n_test = max(
    2,
    round(n * TEST_FRACTION)
)

n_val = max(
    2,
    round(n * VAL_FRACTION)
)

if n - n_test - n_val < 1:
    raise RuntimeError(
        "Invalid train/validation/test split."
    )

test_cases = sorted(
    shuffled[:n_test].tolist()
)

val_cases = sorted(
    shuffled[
        n_test:n_test + n_val
    ].tolist()
)

train_cases = sorted(
    shuffled[
        n_test + n_val:
    ].tolist()
)

print("Train:", len(train_cases), train_cases)
print("Validation:", len(val_cases), val_cases)
print("Test:", len(test_cases), test_cases)

assert set(train_cases).isdisjoint(val_cases)
assert set(train_cases).isdisjoint(test_cases)
assert set(val_cases).isdisjoint(test_cases)

split_rows = []

for case_id in train_cases:
    split_rows.append({
        "case_id": case_id,
        "split": "train"
    })

for case_id in val_cases:
    split_rows.append({
        "case_id": case_id,
        "split": "validation"
    })

for case_id in test_cases:
    split_rows.append({
        "case_id": case_id,
        "split": "test"
    })

pd.DataFrame(split_rows).to_csv(
    SPLIT_CSV,
    index=False
)

print("No case-level leakage.")


# ============================================================
# PREPROCESSING
# ============================================================

def preprocess_ct(volume):
    volume = np.asarray(
        volume,
        dtype=np.float32
    )

    volume = np.nan_to_num(
        volume,
        nan=HU_MIN,
        posinf=HU_MAX,
        neginf=HU_MIN
    )

    volume = np.clip(
        volume,
        HU_MIN,
        HU_MAX
    )

    volume = (
        (volume - HU_MIN) /
        (HU_MAX - HU_MIN)
    )

    volume = (
        volume * 2.0
    ) - 1.0

    return volume.astype(
        np.float32,
        copy=False
    )


def calculate_resampled_shape(
    shape,
    spacing,
    target_spacing
):
    shape = np.asarray(shape, dtype=np.float64)
    spacing = np.asarray(spacing, dtype=np.float64)
    target_spacing = np.asarray(
        target_spacing,
        dtype=np.float64
    )

    new_shape = np.round(
        shape * spacing / target_spacing
    ).astype(int)

    new_shape = np.maximum(
        new_shape,
        1
    )

    return tuple(
        int(v) for v in new_shape
    )


def load_case(
    image_path,
    label_path,
    target_spacing=TARGET_SPACING
):
    image_nii = nib.load(
        str(image_path)
    )

    label_nii = nib.load(
        str(label_path)
    )

    original_spacing = tuple(
        np.asarray(
            image_nii.header.get_zooms()
        )[:3]
    )

    original_shape = tuple(
        image_nii.shape
    )

    target_shape = calculate_resampled_shape(
        original_shape,
        original_spacing,
        target_spacing
    )

    target = (
        target_shape,
        np.eye(4)
    )

    # Use the original affine for physical orientation.
    # The target shape is determined from physical spacing.
    target_affine = np.array(
        image_nii.affine,
        dtype=np.float64
    )

    direction = target_affine[:3, :3].copy()

    norms = np.linalg.norm(
        direction,
        axis=0
    )

    norms[norms == 0] = 1.0

    direction = direction / norms

    target_affine[:3, :3] = (
        direction *
        np.asarray(target_spacing)[None, :]
    )

    target = (
        target_shape,
        target_affine
    )

    if (
        tuple(original_spacing)
        !=
        tuple(target_spacing)
    ):

        image_resampled = resample_from_to(
            image_nii,
            target,
            order=1
        )

        label_resampled = resample_from_to(
            label_nii,
            target,
            order=0
        )

        image = np.asarray(
            image_resampled.dataobj,
            dtype=np.float32
        )

        label = np.asarray(
            label_resampled.dataobj,
            dtype=np.uint8
        )

        affine = np.asarray(
            image_resampled.affine
        )

        del image_resampled
        del label_resampled

    else:

        image = np.asarray(
            image_nii.dataobj,
            dtype=np.float32
        )

        label = np.asarray(
            label_nii.dataobj,
            dtype=np.uint8
        )

        affine = np.asarray(
            image_nii.affine
        )

    image = preprocess_ct(image)

    label = (
        label > 0
    ).astype(np.float32)

    del image_nii
    del label_nii

    return (
        image,
        label,
        affine,
        tuple(target_spacing)
    )


# ============================================================
# PATCH UTILITIES
# ============================================================

def pad_volume(
    volume,
    patch_size,
    constant_value
):
    pz, py, px = patch_size

    pad_z = max(
        0,
        pz - volume.shape[0]
    )

    pad_y = max(
        0,
        py - volume.shape[1]
    )

    pad_x = max(
        0,
        px - volume.shape[2]
    )

    if pad_z or pad_y or pad_x:
        volume = np.pad(
            volume,
            (
                (0, pad_z),
                (0, pad_y),
                (0, pad_x)
            ),
            mode="constant",
            constant_values=constant_value
        )

    return volume


def extract_patch(
    image,
    label,
    start,
    patch_size=PATCH_SIZE
):
    z, y, x = start
    pz, py, px = patch_size

    image_patch = image[
        z:z + pz,
        y:y + py,
        x:x + px
    ]

    label_patch = label[
        z:z + pz,
        y:y + py,
        x:x + px
    ]

    image_patch = pad_volume(
        image_patch,
        patch_size,
        -1.0
    )

    label_patch = pad_volume(
        label_patch,
        patch_size,
        0.0
    )

    return (
        image_patch,
        label_patch
    )


def random_start(
    shape,
    patch_size
):
    return tuple(
        int(
            np.random.randint(
                0,
                max(
                    1,
                    shape[i] - patch_size[i] + 1
                )
            )
        )
        for i in range(3)
    )


def foreground_start(
    label,
    patch_size
):
    foreground = np.argwhere(
        label > 0
    )

    if len(foreground) == 0:
        return random_start(
            label.shape,
            patch_size
        )

    center = foreground[
        np.random.randint(
            len(foreground)
        )
    ]

    starts = []

    for axis in range(3):
        max_start = max(
            0,
            label.shape[axis] -
            patch_size[axis]
        )

        start = int(
            np.clip(
                center[axis] -
                patch_size[axis] // 2,
                0,
                max_start
            )
        )

        starts.append(start)

    return tuple(starts)


def hard_negative_start(
    label,
    patch_size
):
    foreground = label > 0

    if not foreground.any():
        return random_start(
            label.shape,
            patch_size
        )

    distance = distance_transform_edt(
        ~foreground
    )

    candidates = np.argwhere(
        (
            distance >=
            HARD_NEGATIVE_MIN_DISTANCE
        )
        &
        (
            distance <=
            HARD_NEGATIVE_MAX_DISTANCE
        )
    )

    if len(candidates) == 0:
        return random_start(
            label.shape,
            patch_size
        )

    center = candidates[
        np.random.randint(
            len(candidates)
        )
    ]

    starts = []

    for axis in range(3):
        max_start = max(
            0,
            label.shape[axis] -
            patch_size[axis]
        )

        start = int(
            np.clip(
                center[axis] -
                patch_size[axis] // 2,
                0,
                max_start
            )
        )

        starts.append(start)

    return tuple(starts)


def patch_foreground_ratio(
    label_patch
):
    return float(
        np.mean(label_patch > 0)
    )


def sample_training_patch(
    image,
    label
):
    random_value = np.random.random()

    if (
        random_value
        <
        FOREGROUND_PATCH_FRACTION
    ):
        for _ in range(10):
            start = foreground_start(
                label,
                PATCH_SIZE
            )

            image_patch, label_patch = (
                extract_patch(
                    image,
                    label,
                    start
                )
            )

            if (
                patch_foreground_ratio(
                    label_patch
                )
                >=
                MIN_FOREGROUND_RATIO
            ):
                return (
                    image_patch,
                    label_patch
                )

    elif (
        random_value
        <
        (
            FOREGROUND_PATCH_FRACTION
            +
            HARD_NEGATIVE_PATCH_FRACTION
        )
    ):
        start = hard_negative_start(
            label,
            PATCH_SIZE
        )

        return extract_patch(
            image,
            label,
            start
        )

    start = random_start(
        image.shape,
        PATCH_SIZE
    )

    return extract_patch(
        image,
        label,
        start
    )


def augment_patch(
    image_patch,
    label_patch
):
    # All augmentations are geometric and therefore applied
    # identically to image and label.

    if np.random.random() < 0.5:
        image_patch = image_patch[::-1].copy()
        label_patch = label_patch[::-1].copy()

    if np.random.random() < 0.5:
        image_patch = image_patch[:, ::-1].copy()
        label_patch = label_patch[:, ::-1].copy()

    if np.random.random() < 0.5:
        image_patch = image_patch[:, :, ::-1].copy()
        label_patch = label_patch[:, :, ::-1].copy()

    return (
        image_patch,
        label_patch
    )


def make_training_patch_batch(
    image,
    label,
    batch_size=PATCH_BATCH_SIZE
):
    images = []
    labels = []

    for _ in range(batch_size):
        image_patch, label_patch = (
            sample_training_patch(
                image,
                label
            )
        )

        image_patch, label_patch = (
            augment_patch(
                image_patch,
                label_patch
            )
        )

        images.append(image_patch)
        labels.append(label_patch)

    images = np.stack(
        images,
        axis=0
    )

    labels = np.stack(
        labels,
        axis=0
    )

    images = torch.from_numpy(
        images
    ).unsqueeze(1).float()

    labels = torch.from_numpy(
        labels
    ).unsqueeze(1).float()

    return (
        images,
        labels
    )


# ============================================================
# MODEL
# ============================================================

class ConvBlock3D(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):
        super().__init__()

        # IMPORTANT:
        # inplace=False is required for reliable Grad-CAM
        # backward hooks.
        self.block = nn.Sequential(
            nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True
            ),
            nn.LeakyReLU(
                0.01,
                inplace=False
            ),
            nn.Conv3d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True
            ),
            nn.LeakyReLU(
                0.01,
                inplace=False
            )
        )

    def forward(self, x):
        return self.block(x)


class Small3DUNet(nn.Module):

    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        base=16
    ):
        super().__init__()

        self.enc1 = ConvBlock3D(
            in_channels,
            base
        )

        self.pool1 = nn.MaxPool3d(2)

        self.enc2 = ConvBlock3D(
            base,
            base * 2
        )

        self.pool2 = nn.MaxPool3d(2)

        self.enc3 = ConvBlock3D(
            base * 2,
            base * 4
        )

        self.pool3 = nn.MaxPool3d(2)

        self.bottleneck = ConvBlock3D(
            base * 4,
            base * 8
        )

        self.up3 = nn.ConvTranspose3d(
            base * 8,
            base * 4,
            kernel_size=2,
            stride=2
        )

        self.dec3 = ConvBlock3D(
            base * 8,
            base * 4
        )

        self.up2 = nn.ConvTranspose3d(
            base * 4,
            base * 2,
            kernel_size=2,
            stride=2
        )

        self.dec2 = ConvBlock3D(
            base * 4,
            base * 2
        )

        self.up1 = nn.ConvTranspose3d(
            base * 2,
            base,
            kernel_size=2,
            stride=2
        )

        self.dec1 = ConvBlock3D(
            base * 2,
            base
        )

        self.out = nn.Conv3d(
            base,
            out_channels,
            kernel_size=1
        )

    def forward(self, x):

        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool1(e1)
        )

        e3 = self.enc3(
            self.pool2(e2)
        )

        b = self.bottleneck(
            self.pool3(e3)
        )

        d3 = self.up3(b)

        d3 = torch.cat(
            [d3, e3],
            dim=1
        )

        d3 = self.dec3(d3)

        d2 = self.up2(d3)

        d2 = torch.cat(
            [d2, e2],
            dim=1
        )

        d2 = self.dec2(d2)

        d1 = self.up1(d2)

        d1 = torch.cat(
            [d1, e1],
            dim=1
        )

        d1 = self.dec1(d1)

        return self.out(d1)


# ============================================================
# LOSS
# ============================================================

class DiceLoss(nn.Module):

    def __init__(
        self,
        smooth=1.0
    ):
        super().__init__()
        self.smooth = smooth

    def forward(
        self,
        logits,
        targets
    ):
        probs = torch.sigmoid(logits)

        probs = probs.reshape(
            probs.size(0),
            -1
        )

        targets = targets.reshape(
            targets.size(0),
            -1
        )

        intersection = (
            probs * targets
        ).sum(dim=1)

        denominator = (
            probs.sum(dim=1)
            +
            targets.sum(dim=1)
        )

        dice = (
            2.0 * intersection
            +
            self.smooth
        ) / (
            denominator
            +
            self.smooth
        )

        return 1.0 - dice.mean()


class TverskyLoss(nn.Module):

    def __init__(
        self,
        alpha=0.7,
        beta=0.3,
        smooth=1.0
    ):
        super().__init__()

        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth

    def forward(
        self,
        logits,
        targets
    ):
        probs = torch.sigmoid(logits)

        probs = probs.reshape(
            probs.size(0),
            -1
        )

        targets = targets.reshape(
            targets.size(0),
            -1
        )

        tp = (
            probs * targets
        ).sum(dim=1)

        fp = (
            probs *
            (1.0 - targets)
        ).sum(dim=1)

        fn = (
            (1.0 - probs) *
            targets
        ).sum(dim=1)

        tversky = (
            tp + self.smooth
        ) / (
            tp
            +
            self.alpha * fp
            +
            self.beta * fn
            +
            self.smooth
        )

        return 1.0 - tversky.mean()


dice_loss = DiceLoss()

tversky_loss = TverskyLoss(
    alpha=0.7,
    beta=0.3
)

bce_loss = nn.BCEWithLogitsLoss()


def segmentation_loss(
    logits,
    targets
):
    return (
        0.40 * dice_loss(
            logits,
            targets
        )
        +
        0.40 * tversky_loss(
            logits,
            targets
        )
        +
        0.20 * bce_loss(
            logits,
            targets
        )
    )


# ============================================================
# METRICS
# ============================================================

def dice_score(
    prediction,
    target,
    eps=1e-7
):
    prediction = prediction.astype(bool)
    target = target.astype(bool)

    intersection = np.logical_and(
        prediction,
        target
    ).sum()

    return (
        2.0 * intersection + eps
    ) / (
        prediction.sum()
        +
        target.sum()
        +
        eps
    )


def iou_score(
    prediction,
    target,
    eps=1e-7
):
    prediction = prediction.astype(bool)
    target = target.astype(bool)

    intersection = np.logical_and(
        prediction,
        target
    ).sum()

    union = np.logical_or(
        prediction,
        target
    ).sum()

    return (
        intersection + eps
    ) / (
        union + eps
    )


def sensitivity_score(
    prediction,
    target,
    eps=1e-7
):
    prediction = prediction.astype(bool)
    target = target.astype(bool)

    tp = np.logical_and(
        prediction,
        target
    ).sum()

    fn = np.logical_and(
        ~prediction,
        target
    ).sum()

    return (
        tp + eps
    ) / (
        tp + fn + eps
    )


def precision_score(
    prediction,
    target,
    eps=1e-7
):
    prediction = prediction.astype(bool)
    target = target.astype(bool)

    tp = np.logical_and(
        prediction,
        target
    ).sum()

    fp = np.logical_and(
        prediction,
        ~target
    ).sum()

    return (
        tp + eps
    ) / (
        tp + fp + eps
    )


def hd95_mm(
    prediction,
    target,
    spacing
):
    prediction = prediction.astype(bool)
    target = target.astype(bool)

    if (
        not prediction.any()
        or
        not target.any()
    ):
        return np.nan

    pred_surface = (
        prediction
        ^
        binary_erosion(
            prediction
        )
    )

    target_surface = (
        target
        ^
        binary_erosion(
            target
        )
    )

    if (
        not pred_surface.any()
        or
        not target_surface.any()
    ):
        return np.nan

    spacing = tuple(
        float(x)
        for x in spacing
    )

    dt_pred = distance_transform_edt(
        ~prediction,
        sampling=spacing
    )

    dt_target = distance_transform_edt(
        ~target,
        sampling=spacing
    )

    distances_1 = dt_target[
        pred_surface
    ]

    distances_2 = dt_pred[
        target_surface
    ]

    distances = np.concatenate([
        distances_1,
        distances_2
    ])

    return float(
        np.percentile(
            distances,
            95
        )
    )


# ============================================================
# INFERENCE STARTS
# ============================================================

def compute_starts(
    length,
    patch,
    overlap
):
    if length <= patch:
        return [0]

    stride = max(
        1,
        int(
            patch *
            (1.0 - overlap)
        )
    )

    starts = list(
        range(
            0,
            length - patch + 1,
            stride
        )
    )

    last = length - patch

    if starts[-1] != last:
        starts.append(last)

    return starts


def sliding_window_predict(
    model,
    volume,
    patch_size=INFERENCE_PATCH,
    overlap=INFERENCE_OVERLAP,
    batch_size=INFERENCE_BATCH_SIZE
):
    model.eval()

    original_shape = volume.shape

    padded = pad_volume(
        volume,
        patch_size,
        -1.0
    )

    padded_shape = padded.shape

    probability = np.zeros(
        padded_shape,
        dtype=np.float32
    )

    count_map = np.zeros(
        padded_shape,
        dtype=np.float32
    )

    pz, py, px = patch_size

    zs = compute_starts(
        padded_shape[0],
        pz,
        overlap
    )

    ys = compute_starts(
        padded_shape[1],
        py,
        overlap
    )

    xs = compute_starts(
        padded_shape[2],
        px,
        overlap
    )

    starts = [
        (z, y, x)
        for z in zs
        for y in ys
        for x in xs
    ]

    total_windows = len(starts)

    print(
        f"Sliding-window inference: "
        f"{total_windows} windows"
    )

    model_patches = []
    model_starts = []

    with torch.no_grad():

        for index, start in enumerate(
            starts,
            start=1
        ):

            z, y, x = start

            patch = padded[
                z:z+pz,
                y:y+py,
                x:x+px
            ]

            model_patches.append(
                patch
            )

            model_starts.append(
                start
            )

            if (
                len(model_patches)
                ==
                batch_size
                or
                index == total_windows
            ):

                batch = np.stack(
                    model_patches,
                    axis=0
                )

                batch = torch.from_numpy(
                    batch
                ).unsqueeze(1).float().to(
                    DEVICE,
                    non_blocking=True
                )

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=AMP_ENABLED
                ):

                    logits = model(batch)

                    probs = torch.sigmoid(
                        logits
                    )

                probs = (
                    probs[:, 0]
                    .float()
                    .cpu()
                    .numpy()
                )

                for local_index, start_local in enumerate(
                    model_starts
                ):

                    z0, y0, x0 = start_local

                    probability[
                        z0:z0+pz,
                        y0:y0+py,
                        x0:x0+px
                    ] += probs[
                        local_index
                    ]

                    count_map[
                        z0:z0+pz,
                        y0:y0+py,
                        x0:x0+px
                    ] += 1.0

                del batch
                del logits
                del probs

                model_patches.clear()
                model_starts.clear()

                if (
                    index == batch_size
                    or
                    index % 100 == 0
                    or
                    index == total_windows
                ):
                    print(
                        f"Inference "
                        f"{index}/{total_windows}"
                    )

    probability /= np.maximum(
        count_map,
        1e-7
    )

    probability = probability[
        :original_shape[0],
        :original_shape[1],
        :original_shape[2]
    ]

    del padded
    del count_map

    clear_memory()

    return probability.astype(
        np.float32
    )


# ============================================================
# CASE-LEVEL FULL-VOLUME EVALUATION
# ============================================================

def evaluate_case_probability(
    probability,
    target,
    spacing,
    threshold
):
    prediction = (
        probability >= threshold
    )

    return {
        "dice": dice_score(
            prediction,
            target
        ),
        "iou": iou_score(
            prediction,
            target
        ),
        "sensitivity": sensitivity_score(
            prediction,
            target
        ),
        "precision": precision_score(
            prediction,
            target
        ),
        "hd95_mm": hd95_mm(
            prediction,
            target,
            spacing
        )
    }


def load_resampled_case(
    case_id
):
    row = index_df[
        index_df.case_id == case_id
    ].iloc[0]

    image, label, affine, spacing = load_case(
        row["image"],
        row["label"],
        TARGET_SPACING
    )

    return (
        image,
        label,
        affine,
        spacing,
        row
    )


# ============================================================
# MODEL BUILD
# ============================================================

print("\n" + "=" * 80)
print("BUILDING MODEL")
print("=" * 80)

model = Small3DUNet(
    in_channels=1,
    out_channels=1,
    base=BASE_CHANNELS
).to(DEVICE)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Architecture: 3D U-Net")
print("Patch:", PATCH_SIZE)
print("Base channels:", BASE_CHANNELS)
print("Parameters:", f"{total_params:,}")
print("Trainable:", f"{trainable_params:,}")
print("Device:", next(model.parameters()).device)

gpu_memory()


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2
    )
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)


# ============================================================
# VALIDATION CACHE
#
# Only validation probability maps are retained temporarily
# for threshold selection. They are not retained during
# training. Three cases are small enough for this.
# ============================================================

def full_volume_validation(
    model,
    case_list,
    threshold=None,
    return_probabilities=False
):
    model.eval()

    rows = []
    probabilities = {}

    for case_id in case_list:

        print(
            "\nValidation case:",
            case_id
        )

        image, label, affine, spacing, row = (
            load_resampled_case(case_id)
        )

        probability = sliding_window_predict(
            model,
            image
        )

        if threshold is None:
            current_threshold = DEFAULT_THRESHOLD
        else:
            current_threshold = threshold

        metrics = evaluate_case_probability(
            probability,
            label,
            spacing,
            current_threshold
        )

        rows.append({
            "case_id": case_id,
            "threshold": current_threshold,
            **metrics
        })

        if return_probabilities:
            probabilities[case_id] = (
                probability.copy()
            )

        del image
        del label
        del affine
        del probability

        clear_memory()

    metrics_df = pd.DataFrame(rows)

    return (
        metrics_df,
        probabilities
    )


# ============================================================
# THRESHOLD SEARCH
# ============================================================

def search_validation_threshold(
    probability_maps,
    labels,
    spacings
):
    rows = []

    for threshold in THRESHOLD_GRID:

        case_dice = []
        case_precision = []
        case_sensitivity = []
        case_iou = []

        for case_id in probability_maps:

            probability = probability_maps[
                case_id
            ]

            target = labels[
                case_id
            ]

            spacing = spacings[
                case_id
            ]

            metrics = evaluate_case_probability(
                probability,
                target,
                spacing,
                float(threshold)
            )

            case_dice.append(
                metrics["dice"]
            )

            case_precision.append(
                metrics["precision"]
            )

            case_sensitivity.append(
                metrics["sensitivity"]
            )

            case_iou.append(
                metrics["iou"]
            )

        rows.append({
            "threshold": float(threshold),
            "mean_dice": float(
                np.mean(case_dice)
            ),
            "mean_iou": float(
                np.mean(case_iou)
            ),
            "mean_sensitivity": float(
                np.mean(case_sensitivity)
            ),
            "mean_precision": float(
                np.mean(case_precision)
            )
        })

    threshold_df = pd.DataFrame(rows)

    best_row = threshold_df.loc[
        threshold_df["mean_dice"].idxmax()
    ]

    best_threshold = float(
        best_row["threshold"]
    )

    return (
        best_threshold,
        threshold_df
    )


# ============================================================
# TRAINING
# ============================================================

print("\n" + "=" * 80)
print("TRAINING")
print("=" * 80)

print("Train cases:", train_cases)
print("Validation cases:", val_cases)
print("Patch:", PATCH_SIZE)
print("Patch batch:", PATCH_BATCH_SIZE)
print("Epochs:", EPOCHS)
print("AMP:", AMP_ENABLED)
print("Target spacing:", TARGET_SPACING)

best_val_dice = -np.inf
best_threshold = DEFAULT_THRESHOLD
patience_counter = 0

history = []

training_start = time.time()


for epoch in range(
    1,
    EPOCHS + 1
):

    epoch_start = time.time()

    model.train()

    train_losses = []

    print(
        "\n" + "-" * 80
    )

    print(
        f"EPOCH {epoch}/{EPOCHS}"
    )

    # --------------------------------------------------------
    # TRAIN CASE BY CASE
    #
    # Each case is loaded exactly once for the epoch.
    # --------------------------------------------------------

    shuffled_train_cases = (
        train_cases.copy()
    )

    random.shuffle(
        shuffled_train_cases
    )

    for case_index, case_id in enumerate(
        shuffled_train_cases,
        start=1
    ):

        print(
            f"\nTrain case "
            f"{case_id} "
            f"({case_index}/{len(train_cases)})"
        )

        image, label, affine, spacing, row = (
            load_resampled_case(case_id)
        )

        for patch_index in range(
            TRAIN_PATCHES_PER_CASE
        ):

            images, masks = (
                make_training_patch_batch(
                    image,
                    label,
                    PATCH_BATCH_SIZE
                )
            )

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            masks = masks.to(
                DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=AMP_ENABLED
            ):

                logits = model(images)

                loss = segmentation_loss(
                    logits,
                    masks
                )

            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            train_losses.append(
                float(loss.item())
            )

            if (
                patch_index == 1
                or
                patch_index %
                max(
                    1,
                    TRAIN_PATCHES_PER_CASE // 4
                )
                == 0
                or
                patch_index ==
                TRAIN_PATCHES_PER_CASE
            ):

                print(
                    f"Case {case_id} | "
                    f"Patch "
                    f"{patch_index}/"
                    f"{TRAIN_PATCHES_PER_CASE} | "
                    f"Loss "
                    f"{loss.item():.5f}"
                )

            del images
            del masks
            del logits
            del loss

        del image
        del label
        del affine

        clear_memory()

    train_loss = float(
        np.mean(train_losses)
    )

    # --------------------------------------------------------
    # FULL-VOLUME VALIDATION
    #
    # This is now the SAME inference distribution as test.
    # --------------------------------------------------------

    print(
        "\nFULL-VOLUME VALIDATION"
    )

    validation_probability_maps = {}
    validation_labels = {}
    validation_spacings = {}

    model.eval()

    for case_id in val_cases:

        print(
            "\nValidation case:",
            case_id
        )

        image, label, affine, spacing, row = (
            load_resampled_case(case_id)
        )

        probability = sliding_window_predict(
            model,
            image
        )

        validation_probability_maps[
            case_id
        ] = probability.copy()

        validation_labels[
            case_id
        ] = label.copy()

        validation_spacings[
            case_id
        ] = spacing

        del image
        del label
        del affine
        del probability

        clear_memory()

    best_threshold_epoch, threshold_df = (
        search_validation_threshold(
            validation_probability_maps,
            validation_labels,
            validation_spacings
        )
    )

    threshold_df.to_csv(
        RESULT_DIR /
        f"threshold_search_epoch_{epoch:02d}.csv",
        index=False
    )

    current_threshold = (
        best_threshold_epoch
    )

    validation_rows = []

    for case_id in val_cases:

        metrics = evaluate_case_probability(
            validation_probability_maps[
                case_id
            ],
            validation_labels[
                case_id
            ],
            validation_spacings[
                case_id
            ],
            current_threshold
        )

        validation_rows.append({
            "epoch": epoch,
            "case_id": case_id,
            "threshold": current_threshold,
            **metrics
        })

    validation_df = pd.DataFrame(
        validation_rows
    )

    validation_df.to_csv(
        VAL_METRICS_CSV,
        mode="a",
        header=not VAL_METRICS_CSV.exists(),
        index=False
    )

    val_dice = float(
        validation_df["dice"].mean()
    )

    val_iou = float(
        validation_df["iou"].mean()
    )

    val_sensitivity = float(
        validation_df["sensitivity"].mean()
    )

    val_precision = float(
        validation_df["precision"].mean()
    )

    val_hd95 = float(
        validation_df["hd95_mm"].mean()
    )

    scheduler.step(
        val_dice
    )

    current_lr = (
        optimizer.param_groups[0]["lr"]
    )

    epoch_time = (
        time.time()
        -
        epoch_start
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_dice": val_dice,
        "val_iou": val_iou,
        "val_sensitivity": val_sensitivity,
        "val_precision": val_precision,
        "val_hd95_mm": val_hd95,
        "threshold": current_threshold,
        "learning_rate": current_lr,
        "time_sec": epoch_time
    })

    print(
        "\n" + "=" * 80
    )

    print(
        f"Epoch {epoch}/{EPOCHS}"
    )

    print(
        f"Train Loss       : {train_loss:.5f}"
    )

    print(
        f"Val Dice         : {val_dice:.5f}"
    )

    print(
        f"Val IoU          : {val_iou:.5f}"
    )

    print(
        f"Val Sensitivity  : {val_sensitivity:.5f}"
    )

    print(
        f"Val Precision    : {val_precision:.5f}"
    )

    print(
        f"Val HD95 (mm)    : {val_hd95:.3f}"
    )

    print(
        f"Val Threshold    : {current_threshold:.2f}"
    )

    print(
        f"Learning rate    : {current_lr:.2e}"
    )

    print(
        f"Epoch time       : {epoch_time / 60:.2f} min"
    )

    gpu_memory()

    # --------------------------------------------------------
    # CHECKPOINT
    # --------------------------------------------------------

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_dice": best_val_dice,
        "current_val_dice": val_dice,
        "selected_threshold": current_threshold,
        "config": {
            "patch_size": PATCH_SIZE,
            "base_channels": BASE_CHANNELS,
            "target_spacing": TARGET_SPACING,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "foreground_patch_fraction":
                FOREGROUND_PATCH_FRACTION,
            "hard_negative_patch_fraction":
                HARD_NEGATIVE_PATCH_FRACTION,
            "background_patch_fraction":
                BACKGROUND_PATCH_FRACTION,
        }
    }

    torch.save(
        checkpoint,
        LATEST_CHECKPOINT
    )

    # --------------------------------------------------------
    # BEST MODEL
    # --------------------------------------------------------

    if val_dice > best_val_dice:

        best_val_dice = val_dice
        best_threshold = current_threshold
        patience_counter = 0

        checkpoint[
            "best_val_dice"
        ] = best_val_dice

        checkpoint[
            "selected_threshold"
        ] = best_threshold

        torch.save(
            checkpoint,
            BEST_CHECKPOINT
        )

        print(
            "\nBEST CHECKPOINT SAVED"
        )

        print(
            "Best validation Dice:",
            f"{best_val_dice:.5f}"
        )

        print(
            "Selected threshold:",
            f"{best_threshold:.2f}"
        )

    else:

        patience_counter += 1

        print(
            f"\nNo improvement | "
            f"patience "
            f"{patience_counter}/{PATIENCE}"
        )

    # --------------------------------------------------------
    # RELEASE VALIDATION ARRAYS
    # --------------------------------------------------------

    del validation_probability_maps
    del validation_labels
    del validation_spacings

    clear_memory()

    if patience_counter >= PATIENCE:

        print(
            "\nEarly stopping."
        )

        break


training_time = (
    time.time()
    -
    training_start
)

history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    HISTORY_CSV,
    index=False
)

print(
    "\nTraining completed in",
    f"{training_time / 3600:.2f} hours"
)


# ============================================================
# LOAD BEST MODEL
# ============================================================

print("\n" + "=" * 80)
print("LOADING BEST CHECKPOINT")
print("=" * 80)

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

best_val_dice = float(
    checkpoint["best_val_dice"]
)

best_threshold = float(
    checkpoint["selected_threshold"]
)

print(
    "Best validation Dice:",
    f"{best_val_dice:.5f}"
)

print(
    "Selected validation threshold:",
    f"{best_threshold:.2f}"
)

print(
    "Checkpoint:",
    BEST_CHECKPOINT
)

clear_memory()


# ============================================================
# FINAL TEST
#
# IMPORTANT:
# Threshold is frozen from validation.
# It is NOT optimized using test labels.
# ============================================================

print("\n" + "=" * 80)
print("FINAL TEST")
print("=" * 80)

print(
    "Frozen threshold:",
    f"{best_threshold:.2f}"
)

test_results = []

for test_index, case_id in enumerate(
    test_cases,
    start=1
):

    print(
        "\n" + "-" * 80
    )

    print(
        f"TEST CASE {case_id} "
        f"({test_index}/{len(test_cases)})"
    )

    image, label, affine, spacing, row = (
        load_resampled_case(case_id)
    )

    print(
        "Resampled volume:",
        image.shape
    )

    print(
        "Spacing:",
        spacing
    )

    probability = sliding_window_predict(
        model,
        image
    )

    metrics = evaluate_case_probability(
        probability,
        label,
        spacing,
        best_threshold
    )

    prediction = (
        probability >= best_threshold
    )

    result_row = {
        "case_id": case_id,
        "threshold": best_threshold,
        **metrics
    }

    test_results.append(
        result_row
    )

    print(
        f"Dice       : {metrics['dice']:.5f}"
    )

    print(
        f"IoU        : {metrics['iou']:.5f}"
    )

    print(
        f"Sensitivity: {metrics['sensitivity']:.5f}"
    )

    print(
        f"Precision  : {metrics['precision']:.5f}"
    )

    print(
        f"HD95 (mm)  : {metrics['hd95_mm']}"
    )

    # --------------------------------------------------------
    # Save outputs in RESAMPLED physical space.
    # --------------------------------------------------------

    probability_nii = nib.Nifti1Image(
        probability.astype(np.float32),
        affine
    )

    prediction_nii = nib.Nifti1Image(
        prediction.astype(np.uint8),
        affine
    )

    probability_path = (
        RESULT_DIR /
        f"case_{case_id}_probability_resampled.nii.gz"
    )

    prediction_path = (
        RESULT_DIR /
        f"case_{case_id}_prediction_resampled.nii.gz"
    )

    nib.save(
        probability_nii,
        str(probability_path)
    )

    nib.save(
        prediction_nii,
        str(prediction_path)
    )

    print(
        "Saved:",
        probability_path
    )

    print(
        "Saved:",
        prediction_path
    )

    del image
    del label
    del affine
    del probability
    del prediction
    del probability_nii
    del prediction_nii

    clear_memory()


test_df = pd.DataFrame(
    test_results
)

test_df.to_csv(
    TEST_METRICS_CSV,
    index=False
)

print("\n" + "=" * 80)
print("TEST RESULTS")
print("=" * 80)

print(
    test_df.to_string(
        index=False
    )
)

print("\nMEAN TEST METRICS")

for column in [
    "dice",
    "iou",
    "sensitivity",
    "precision",
    "hd95_mm"
]:

    print(
        f"{column:18s}: "
        f"{test_df[column].mean():.5f}"
    )

print("\nSTD TEST METRICS")

for column in [
    "dice",
    "iou",
    "sensitivity",
    "precision",
    "hd95_mm"
]:

    print(
        f"{column:18s}: "
        f"{test_df[column].std(ddof=1):.5f}"
    )


# ============================================================
# 3D GRAD-CAM
#
# Correct implementation:
# 1. No inplace activations
# 2. Uses forward activation hook
# 3. Uses backward gradient hook
# 4. Generates CAM in patch coordinates
# 5. Embeds patch CAM into full resampled volume
# 6. Saves with the full resampled affine
# ============================================================

print("\n" + "=" * 80)
print("3D GRAD-CAM")
print("=" * 80)


class GradCAM3D:

    def __init__(
        self,
        model,
        target_layer
    ):
        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        self.forward_handle = (
            target_layer.register_forward_hook(
                self._forward_hook
            )
        )

        self.backward_handle = (
            target_layer.register_full_backward_hook(
                self._backward_hook
            )
        )

    def _forward_hook(
        self,
        module,
        inputs,
        output
    ):
        self.activations = output

    def _backward_hook(
        self,
        module,
        grad_input,
        grad_output
    ):
        self.gradients = grad_output[0]

    def generate(
        self,
        patch
    ):
        self.model.zero_grad(
            set_to_none=True
        )

        self.activations = None
        self.gradients = None

        tensor = (
            torch.from_numpy(
                patch
            )
            .unsqueeze(0)
            .unsqueeze(0)
            .float()
            .to(DEVICE)
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=AMP_ENABLED
        ):

            logits = self.model(
                tensor
            )

            probability = torch.sigmoid(
                logits
            )

            # Positive-class objective.
            # Use the mean foreground probability so that the
            # objective is stable even when the prediction is
            # sparse.
            objective = probability.mean()

        objective.backward()

        if (
            self.activations is None
            or
            self.gradients is None
        ):
            raise RuntimeError(
                "Grad-CAM activation or gradient unavailable."
            )

        gradients = self.gradients.float()
        activations = self.activations.float()

        weights = gradients.mean(
            dim=(2, 3, 4),
            keepdim=True
        )

        cam = (
            weights *
            activations
        ).sum(
            dim=1,
            keepdim=True
        )

        cam = F.relu(cam)

        cam = F.interpolate(
            cam,
            size=patch.shape,
            mode="trilinear",
            align_corners=False
        )

        cam = cam[
            0,
            0
        ]

        cam_min = cam.min()
        cam_max = cam.max()

        cam = (
            cam - cam_min
        ) / (
            cam_max -
            cam_min +
            1e-8
        )

        cam = (
            cam
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        probability = (
            probability[
                0,
                0
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        del tensor
        del logits
        del probability
        del objective
        del gradients
        del activations

        clear_memory()

        return cam

    def close(self):
        self.forward_handle.remove()
        self.backward_handle.remove()


if isinstance(model, nn.DataParallel):
    xai_model = model.module
else:
    xai_model = model

target_layer = (
    xai_model
    .enc3
    .block[-1]
)

gradcam = GradCAM3D(
    xai_model,
    target_layer
)


# ============================================================
# XAI CASES
# ============================================================

for case_id in test_cases:

    print(
        "\nXAI case:",
        case_id
    )

    image, label, affine, spacing, row = (
        load_resampled_case(case_id)
    )

    foreground = np.argwhere(
        label > 0
    )

    if len(foreground) == 0:
        print(
            "No foreground. Skipping."
        )

        del image
        del label
        del affine

        continue

    center = foreground[
        len(foreground) // 2
    ]

    pz, py, px = PATCH_SIZE

    z = int(
        np.clip(
            center[0] - pz // 2,
            0,
            max(
                0,
                image.shape[0] - pz
            )
        )
    )

    y = int(
        np.clip(
            center[1] - py // 2,
            0,
            max(
                0,
                image.shape[1] - py
            )
        )
    )

    x = int(
        np.clip(
            center[2] - px // 2,
            0,
            max(
                0,
                image.shape[2] - px
            )
        )
    )

    patch, patch_label = extract_patch(
        image,
        label,
        (z, y, x)
    )

    try:

        cam = gradcam.generate(
            patch
        )

    except Exception as error:

        print(
            "Grad-CAM failed:",
            repr(error)
        )

        del image
        del label
        del affine
        del patch
        del patch_label

        clear_memory()

        continue

    # --------------------------------------------------------
    # Embed patch CAM into the FULL RESAMPLED VOLUME.
    # This fixes the spatial alignment issue in v1.
    # --------------------------------------------------------

    full_cam = np.zeros(
        image.shape,
        dtype=np.float32
    )

    z1 = min(
        z + pz,
        image.shape[0]
    )

    y1 = min(
        y + py,
        image.shape[1]
    )

    x1 = min(
        x + px,
        image.shape[2]
    )

    valid_z = z1 - z
    valid_y = y1 - y
    valid_x = x1 - x

    full_cam[
        z:z1,
        y:y1,
        x:x1
    ] = cam[
        :valid_z,
        :valid_y,
        :valid_x
    ]

    cam_nii = nib.Nifti1Image(
        full_cam,
        affine
    )

    cam_path = (
        XAI_DIR /
        f"case_{case_id}_gradcam_full_resampled.nii.gz"
    )

    nib.save(
        cam_nii,
        str(cam_path)
    )

    # --------------------------------------------------------
    # Center slice visualization.
    # --------------------------------------------------------

    mid = (
        z +
        min(
            pz // 2,
            image.shape[0] - z - 1
        )
    )

    image_slice = image[mid]
    cam_slice = full_cam[mid]
    label_slice = label[mid]

    plt.figure(
        figsize=(15, 5)
    )

    plt.subplot(1, 3, 1)

    plt.imshow(
        image_slice,
        cmap="gray"
    )

    plt.title(
        f"Case {case_id} | CCTA"
    )

    plt.axis("off")

    plt.subplot(1, 3, 2)

    plt.imshow(
        image_slice,
        cmap="gray"
    )

    plt.imshow(
        cam_slice,
        cmap="jet",
        alpha=0.45
    )

    plt.title(
        "3D Grad-CAM"
    )

    plt.axis("off")

    plt.subplot(1, 3, 3)

    plt.imshow(
        image_slice,
        cmap="gray"
    )

    plt.imshow(
        label_slice,
        cmap="Reds",
        alpha=0.45
    )

    plt.title(
        "Ground Truth"
    )

    plt.axis("off")

    plt.tight_layout()

    png_path = (
        XAI_DIR /
        f"case_{case_id}_xai.png"
    )

    plt.savefig(
        png_path,
        dpi=150,
        bbox_inches="tight"
    )

    plt.close()

    print(
        "Saved:",
        cam_path
    )

    print(
        "Saved:",
        png_path
    )

    del image
    del label
    del affine
    del patch
    del patch_label
    del cam
    del full_cam
    del cam_nii

    clear_memory()


gradcam.close()


# ============================================================
# TRAINING CURVES
# ============================================================

if len(history_df) > 0:

    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        history_df["epoch"],
        history_df["train_loss"],
        label="Train Loss"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(
        "CardioVision AI CCA v2 | Training Loss"
    )

    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()

    plt.savefig(
        RESULT_DIR /
        "training_loss.png",
        dpi=150
    )

    plt.close()

    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        history_df["epoch"],
        history_df["val_dice"],
        label="Validation Dice"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Dice")
    plt.title(
        "CardioVision AI CCA v2 | Full-Volume Validation Dice"
    )

    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()

    plt.savefig(
        RESULT_DIR /
        "validation_dice.png",
        dpi=150
    )

    plt.close()

    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        history_df["epoch"],
        history_df["threshold"],
        label="Selected Validation Threshold"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Threshold")
    plt.title(
        "CardioVision AI CCA v2 | Validation Threshold"
    )

    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()

    plt.savefig(
        RESULT_DIR /
        "validation_threshold.png",
        dpi=150
    )

    plt.close()


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 80)
print("CARDIOVISION AI | CCA v2 COMPLETE")
print("=" * 80)

print("\nDATASET")
print("Repository:", DATASET_NAME)
print("Total cases:", len(case_ids))
print("Train:", len(train_cases))
print("Validation:", len(val_cases))
print("Test:", len(test_cases))

print("\nMODEL")
print("Architecture: 3D U-Net")
print("Parameters:", f"{total_params:,}")
print("Base channels:", BASE_CHANNELS)
print("Patch:", PATCH_SIZE)
print("Target spacing:", TARGET_SPACING)
print("AMP:", AMP_ENABLED)

print("\nVALIDATION")
print(
    "Best full-volume validation Dice:",
    f"{best_val_dice:.5f}"
)

print(
    "Frozen validation threshold:",
    f"{best_threshold:.2f}"
)

if len(test_df) > 0:

    print("\nTEST")
    print(
        "Dice:",
        f"{test_df.dice.mean():.5f}"
    )

    print(
        "IoU:",
        f"{test_df.iou.mean():.5f}"
    )

    print(
        "Sensitivity:",
        f"{test_df.sensitivity.mean():.5f}"
    )

    print(
        "Precision:",
        f"{test_df.precision.mean():.5f}"
    )

    print(
        "HD95 (mm):",
        f"{test_df.hd95_mm.mean():.5f}"
    )

print("\nOUTPUTS")
print("Checkpoint:", CHECKPOINT_DIR)
print("Results:", RESULT_DIR)
print("XAI:", XAI_DIR)

print("\nPIPELINE STATUS")
print("✓ CCA dataset acquisition")
print("✓ NIfTI discovery")
print("✓ Image/label validation")
print("✓ Case-level split")
print("✓ No case-level leakage")
print("✓ Physical-space resampling")
print("✓ Foreground-aware sampling")
print("✓ Hard-negative sampling")
print("✓ Background sampling")
print("✓ Case-once-per-epoch loading")
print("✓ 96³ training patches")
print("✓ Dice + Tversky + BCE")
print("✓ AMP FP16")
print("✓ GPU execution")
print("✓ Full-volume validation")
print("✓ Validation threshold optimization")
print("✓ Frozen test threshold")
print("✓ Sliding-window test")
print("✓ Dice / IoU / Sensitivity / Precision")
print("✓ HD95 in millimetres")
print("✓ Spatially aligned 3D Grad-CAM")
print("✓ NIfTI predictions")
print("✓ Training curves")

print("\nCARDIOVISION AI CCA v2 COMPLETE")

CARDIOVISION AI | CCA 3D CCTA SEGMENTATION v2

DEVICE
--------------------------------------------------------------------------------
PyTorch: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
Selected device: cuda:0
GPU count: 2
GPU 0: Tesla T4 | 14.56 GB
GPU 1: Tesla T4 | 14.56 GB

GPU COMPUTE TEST
Tensor device: cuda:0
Model device: cuda:0
Output: (1, 4, 32, 32, 32)
GPU computation verified.

CCA DATASET


Fetching 43 files:   0%|          | 0/43 [00:00<?, ?it/s]

Repository: /root/.cache/huggingface/hub/datasets--MedHK23--CCA/snapshots/a78045d4546ec9f52484920d66152db7f31f84a1

DISCOVERING NIFTI FILES
Images: 20
Labels: 20
Matched cases: 20

VALIDATING DATASET
Validating case 0 (1/20)
Validating case 1 (2/20)
Validating case 2 (3/20)
Validating case 3 (4/20)
Validating case 4 (5/20)
Validating case 5 (6/20)
Validating case 6 (7/20)
Validating case 7 (8/20)
Validating case 8 (9/20)
Validating case 9 (10/20)
Validating case 10 (11/20)
Validating case 11 (12/20)
Validating case 12 (13/20)
Validating case 13 (14/20)
Validating case 14 (15/20)
Validating case 15 (16/20)
Validating case 16 (17/20)
Validating case 17 (18/20)
Validating case 18 (19/20)
Validating case 19 (20/20)

Validation complete.
Index: /kaggle/working/cardioVision_CCA_v2/cardiovision_cca_index.csv
Foreground ratio: 0.000677 to 0.001608

CASE-LEVEL SPLIT
Train: 14 [0, 1, 2, 3, 4, 5, 6, 8, 11, 13, 16, 17, 18, 19]
Validation: 3 [7, 10, 12]
Test: 3 [9, 14, 15]
No case-level leakage.

B